# Webinar 2: Data Preprocessing — Track 1: Tabular Pipeline
### Dataset: Real IBM Telco Customer Churn (7,043 customer records)
### Algorithms: XGBoost & LightGBM vs Baseline Logistic Regression

This notebook covers the complete tabular preprocessing lifecycle on real-world business data:
1. **Profiling**: Missing value audit in `TotalCharges`, extreme outliers in `MonthlyCharges`, skewness, and cardinality.
2. **Encoding**: One-Hot Encoding for nominal columns (`PaymentMethod`, `InternetService`), Ordinal Encoding for `Contract`.
3. **Scaling**: Comparing `StandardScaler`, `MinMaxScaler`, and `RobustScaler` on skewed billing features.
4. **Imbalance**: Handling the ~27% churn minority class using **SMOTE**.
5. **Train Model & Evaluation**: Comparing naive baseline Logistic Regression vs preprocessed **XGBoost Classifier**.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.tabular import (
    profile_dataframe,
    print_profiling_report,
    TabularEncoder,
    TabularScaler,
    compare_scalers,
    balance_dataset,
    run_tabular_pipeline
)

pd.set_option('display.max_columns', None)
print('Imports loaded successfully!')

## Step 1: Data Profiling & Health Audit (IBM Telco Churn)
Inspecting missing values, string anomalies, distributions, and class imbalance.

In [ ]:
df_raw = pd.read_csv('../data/tabular/telco_churn_raw.csv')
print(f'IBM Telco Dataset Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Convert TotalCharges to numeric for profiling
df_profile = df_raw.copy()
df_profile['TotalCharges'] = pd.to_numeric(df_profile['TotalCharges'].astype(str).str.strip(), errors='coerce')
profile = profile_dataframe(df_profile, target_col='Churn')
print_profiling_report(profile)

## Step 2: Categorical Encoding (Nominal vs Ordinal)
- Nominal features (`PaymentMethod`, `InternetService`, etc.) $\to$ **One-Hot Encoding**
- Ordinal feature (`Contract`: Month-to-month < One year < Two year) $\to$ **Ordinal Encoding**

In [ ]:
# Prepare binary and categorical features
X_clean = df_raw.drop(columns=['Churn', 'customerID'])
X_clean['TotalCharges'] = pd.to_numeric(X_clean['TotalCharges'].astype(str).str.strip(), errors='coerce')
y = df_raw['Churn'].map({'Yes': 1, 'No': 0})

nominal_cols = ['PaymentMethod', 'InternetService', 'OnlineSecurity', 'DeviceProtection']
ordinal_cols = ['Contract']
ordinal_order = {'Contract': ['Month-to-month', 'One year', 'Two year']}

encoder = TabularEncoder(
    nominal_cols=nominal_cols,
    ordinal_cols=ordinal_cols,
    ordinal_categories=ordinal_order
)
X_encoded = encoder.fit_transform(X_clean)
print(f'Encoded Feature Matrix Shape: {X_encoded.shape}')
X_encoded.head()

## Step 3: Feature Scaling Comparison (Standard vs MinMax vs Robust)
Comparing scaler resistance to extreme charges.

In [ ]:
scaler_comparison = compare_scalers(X_encoded, num_cols=['MonthlyCharges', 'tenure'])
scaler_comparison

In [ ]:
from src.evaluation.visualizer import plot_tabular_scaling_and_outliers
plot_tabular_scaling_and_outliers(df_profile, num_col='MonthlyCharges', output_path='../reports/tabular_scaling_and_outliers.png')

scaler = TabularScaler(method='robust')
X_scaled = scaler.fit_transform(X_encoded)
X_scaled.head()

## Step 4: Class Imbalance Handling with SMOTE

In [ ]:
print('Original Churn Distribution:')
print(y.value_counts(normalize=True) * 100)

X_balanced, y_balanced = balance_dataset(X_scaled, y, method='smote')
print('\nBalanced Distribution After SMOTE:')
print(y_balanced.value_counts())

## Step 5: Model Training (XGBoost vs Baseline) & Evaluation

In [ ]:
tabular_results = run_tabular_pipeline('../data/tabular/telco_churn_raw.csv')

from src.evaluation.comparison import generate_modality_comparison
comp_df = generate_modality_comparison(tabular_results['baseline_metrics'], tabular_results['preprocessed_metrics'], 'Tabular (Telco Churn)')
comp_df